## TOOL CALLING

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [32]:
MODEL = "gpt-5.4"
openai = OpenAI()

In [40]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

flight_data = {
    "london": {
        "price": "$799",
        "flights": [
            {"flight_no": "BA101", "departure": "06:00", "arrival": "11:30", "duration": "5h 30m", "airline": "British Airways", "stops": 0},
            {"flight_no": "EK302", "departure": "14:00", "arrival": "20:15", "duration": "6h 15m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR211", "departure": "22:30", "arrival": "05:10+1", "duration": "6h 40m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "paris": {
        "price": "$899",
        "flights": [
            {"flight_no": "AF007", "departure": "07:15", "arrival": "13:00", "duration": "5h 45m", "airline": "Air France", "stops": 0},
            {"flight_no": "LH432", "departure": "10:30", "arrival": "17:20", "duration": "6h 50m", "airline": "Lufthansa", "stops": 1},
            {"flight_no": "TK198", "departure": "19:45", "arrival": "02:30+1", "duration": "6h 45m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "tokyo": {
        "price": "$1400",
        "flights": [
            {"flight_no": "NH103", "departure": "10:00", "arrival": "14:30+1", "duration": "13h 30m", "airline": "ANA", "stops": 0},
            {"flight_no": "JL044", "departure": "16:00", "arrival": "20:45+1", "duration": "13h 45m", "airline": "Japan Airlines", "stops": 0},
            {"flight_no": "KE705", "departure": "22:15", "arrival": "06:00+2", "duration": "14h 45m", "airline": "Korean Air", "stops": 1},
        ]
    },
    "berlin": {
        "price": "$499",
        "flights": [
            {"flight_no": "LH100", "departure": "08:00", "arrival": "13:45", "duration": "5h 45m", "airline": "Lufthansa", "stops": 0},
            {"flight_no": "EW205", "departure": "12:30", "arrival": "18:50", "duration": "6h 20m", "airline": "Eurowings", "stops": 1},
            {"flight_no": "FR554", "departure": "20:00", "arrival": "02:30+1", "duration": "6h 30m", "airline": "Ryanair", "stops": 1},
        ]
    },
    "new_york": {
        "price": "$999",
        "flights": [
            {"flight_no": "AA100", "departure": "09:00", "arrival": "17:30", "duration": "8h 30m", "airline": "American Airlines", "stops": 0},
            {"flight_no": "DL401", "departure": "13:45", "arrival": "22:10", "duration": "8h 25m", "airline": "Delta", "stops": 0},
            {"flight_no": "UA318", "departure": "22:00", "arrival": "07:20+1", "duration": "9h 20m", "airline": "United Airlines", "stops": 1},
        ]
    },
    "rome": {
        "price": "$650",
        "flights": [
            {"flight_no": "AZ610", "departure": "07:30", "arrival": "13:30", "duration": "6h 00m", "airline": "Alitalia", "stops": 0},
            {"flight_no": "IB403", "departure": "11:15", "arrival": "18:00", "duration": "6h 45m", "airline": "Iberia", "stops": 1},
            {"flight_no": "TK192", "departure": "21:00", "arrival": "04:30+1", "duration": "7h 30m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "barcelona": {
        "price": "$620",
        "flights": [
            {"flight_no": "VY1234", "departure": "06:45", "arrival": "12:30", "duration": "5h 45m", "airline": "Vueling", "stops": 0},
            {"flight_no": "IB3120", "departure": "14:00", "arrival": "20:15", "duration": "6h 15m", "airline": "Iberia", "stops": 0},
            {"flight_no": "FR8802", "departure": "19:30", "arrival": "02:00+1", "duration": "6h 30m", "airline": "Ryanair", "stops": 1},
        ]
    },
    "amsterdam": {
        "price": "$680",
        "flights": [
            {"flight_no": "KL801", "departure": "08:20", "arrival": "13:50", "duration": "5h 30m", "airline": "KLM", "stops": 0},
            {"flight_no": "EI654", "departure": "15:10", "arrival": "21:00", "duration": "5h 50m", "airline": "Aer Lingus", "stops": 1},
            {"flight_no": "EK162", "departure": "23:00", "arrival": "05:45+1", "duration": "6h 45m", "airline": "Emirates", "stops": 1},
        ]
    },
    "dubai": {
        "price": "$850",
        "flights": [
            {"flight_no": "EK001", "departure": "07:00", "arrival": "16:40", "duration": "9h 40m", "airline": "Emirates", "stops": 0},
            {"flight_no": "FZ3401", "departure": "12:30", "arrival": "22:50", "duration": "10h 20m", "airline": "flydubai", "stops": 1},
            {"flight_no": "EY401", "departure": "20:00", "arrival": "07:15+1", "duration": "11h 15m", "airline": "Etihad", "stops": 1},
        ]
    },
    "singapore": {
        "price": "$920",
        "flights": [
            {"flight_no": "SQ321", "departure": "09:00", "arrival": "22:30", "duration": "13h 30m", "airline": "Singapore Airlines", "stops": 0},
            {"flight_no": "QF001", "departure": "16:00", "arrival": "06:10+1", "duration": "14h 10m", "airline": "Qantas", "stops": 1},
            {"flight_no": "CX771", "departure": "22:30", "arrival": "13:45+1", "duration": "15h 15m", "airline": "Cathay Pacific", "stops": 1},
        ]
    },
    "sydney": {
        "price": "$1200",
        "flights": [
            {"flight_no": "QF001", "departure": "11:00", "arrival": "06:00+2", "duration": "19h 00m", "airline": "Qantas", "stops": 0},
            {"flight_no": "EK413", "departure": "21:30", "arrival": "20:00+2", "duration": "22h 30m", "airline": "Emirates", "stops": 1},
            {"flight_no": "SQ231", "departure": "08:00", "arrival": "05:30+2", "duration": "21h 30m", "airline": "Singapore Airlines", "stops": 1},
        ]
    },
    "los_angeles": {
        "price": "$1050",
        "flights": [
            {"flight_no": "AA200", "departure": "07:00", "arrival": "10:00", "duration": "13h 00m", "airline": "American Airlines", "stops": 0},
            {"flight_no": "UA502", "departure": "13:30", "arrival": "16:55", "duration": "13h 25m", "airline": "United Airlines", "stops": 0},
            {"flight_no": "DL280", "departure": "21:00", "arrival": "01:20+1", "duration": "14h 20m", "airline": "Delta", "stops": 1},
        ]
    },
    "san_francisco": {
        "price": "$1100",
        "flights": [
            {"flight_no": "UA858", "departure": "08:30", "arrival": "11:45", "duration": "13h 15m", "airline": "United Airlines", "stops": 0},
            {"flight_no": "AA136", "departure": "14:00", "arrival": "17:30", "duration": "13h 30m", "airline": "American Airlines", "stops": 0},
            {"flight_no": "BA284", "departure": "20:15", "arrival": "00:10+1", "duration": "13h 55m", "airline": "British Airways", "stops": 1},
        ]
    },
    "las_vegas": {
        "price": "$700",
        "flights": [
            {"flight_no": "SW1102", "departure": "10:00", "arrival": "15:30", "duration": "14h 30m", "airline": "Southwest", "stops": 1},
            {"flight_no": "WN452", "departure": "15:00", "arrival": "21:20", "duration": "15h 20m", "airline": "Southwest", "stops": 1},
            {"flight_no": "F91103", "departure": "22:30", "arrival": "05:45+1", "duration": "16h 15m", "airline": "Frontier", "stops": 1},
        ]
    },
    "toronto": {
        "price": "$880",
        "flights": [
            {"flight_no": "AC826", "departure": "09:00", "arrival": "17:00", "duration": "8h 00m", "airline": "Air Canada", "stops": 0},
            {"flight_no": "BA098", "departure": "14:30", "arrival": "23:10", "duration": "8h 40m", "airline": "British Airways", "stops": 0},
            {"flight_no": "UA630", "departure": "21:00", "arrival": "06:30+1", "duration": "9h 30m", "airline": "United Airlines", "stops": 1},
        ]
    },
    "vancouver": {
        "price": "$900",
        "flights": [
            {"flight_no": "AC556", "departure": "10:15", "arrival": "10:30", "duration": "9h 15m", "airline": "Air Canada", "stops": 0},
            {"flight_no": "BA096", "departure": "16:00", "arrival": "16:45", "duration": "9h 45m", "airline": "British Airways", "stops": 0},
            {"flight_no": "WS022", "departure": "23:00", "arrival": "00:30+1", "duration": "10h 30m", "airline": "WestJet", "stops": 1},
        ]
    },
    "bangkok": {
        "price": "$720",
        "flights": [
            {"flight_no": "TG911", "departure": "08:30", "arrival": "21:00", "duration": "12h 30m", "airline": "Thai Airways", "stops": 0},
            {"flight_no": "EK372", "departure": "14:45", "arrival": "04:30+1", "duration": "13h 45m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR826", "departure": "22:00", "arrival": "13:15+1", "duration": "15h 15m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "hong_kong": {
        "price": "$980",
        "flights": [
            {"flight_no": "CX234", "departure": "10:00", "arrival": "22:30", "duration": "12h 30m", "airline": "Cathay Pacific", "stops": 0},
            {"flight_no": "KA102", "departure": "16:30", "arrival": "05:10+1", "duration": "12h 40m", "airline": "Dragon Air", "stops": 0},
            {"flight_no": "EK382", "departure": "23:30", "arrival": "14:00+1", "duration": "14h 30m", "airline": "Emirates", "stops": 1},
        ]
    },
    "seoul": {
        "price": "$950",
        "flights": [
            {"flight_no": "KE901", "departure": "09:30", "arrival": "23:00", "duration": "13h 30m", "airline": "Korean Air", "stops": 0},
            {"flight_no": "OZ501", "departure": "15:00", "arrival": "05:20+1", "duration": "14h 20m", "airline": "Asiana Airlines", "stops": 0},
            {"flight_no": "SQ607", "departure": "21:45", "arrival": "13:30+1", "duration": "15h 45m", "airline": "Singapore Airlines", "stops": 1},
        ]
    },
    "beijing": {
        "price": "$870",
        "flights": [
            {"flight_no": "CA937", "departure": "11:00", "arrival": "23:50", "duration": "12h 50m", "airline": "Air China", "stops": 0},
            {"flight_no": "MU571", "departure": "17:30", "arrival": "07:00+1", "duration": "13h 30m", "airline": "China Eastern", "stops": 0},
            {"flight_no": "EK304", "departure": "01:00", "arrival": "16:00+1", "duration": "15h 00m", "airline": "Emirates", "stops": 1},
        ]
    },
    "shanghai": {
        "price": "$890",
        "flights": [
            {"flight_no": "MU549", "departure": "10:30", "arrival": "23:15", "duration": "12h 45m", "airline": "China Eastern", "stops": 0},
            {"flight_no": "CZ687", "departure": "16:45", "arrival": "06:20+1", "duration": "13h 35m", "airline": "China Southern", "stops": 0},
            {"flight_no": "QR870", "departure": "23:15", "arrival": "14:45+1", "duration": "15h 30m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "kuala_lumpur": {
        "price": "$740",
        "flights": [
            {"flight_no": "MH004", "departure": "09:15", "arrival": "21:30", "duration": "12h 15m", "airline": "Malaysia Airlines", "stops": 0},
            {"flight_no": "AK524", "departure": "14:00", "arrival": "02:45+1", "duration": "12h 45m", "airline": "AirAsia", "stops": 1},
            {"flight_no": "EK348", "departure": "22:00", "arrival": "12:30+1", "duration": "14h 30m", "airline": "Emirates", "stops": 1},
        ]
    },
    "bali": {
        "price": "$760",
        "flights": [
            {"flight_no": "GA880", "departure": "08:00", "arrival": "22:30", "duration": "14h 30m", "airline": "Garuda Indonesia", "stops": 1},
            {"flight_no": "SQ947", "departure": "13:30", "arrival": "06:00+1", "duration": "16h 30m", "airline": "Singapore Airlines", "stops": 1},
            {"flight_no": "EK362", "departure": "21:00", "arrival": "14:15+1", "duration": "17h 15m", "airline": "Emirates", "stops": 1},
        ]
    },
    "phuket": {
        "price": "$730",
        "flights": [
            {"flight_no": "TG210", "departure": "07:30", "arrival": "21:00", "duration": "13h 30m", "airline": "Thai Airways", "stops": 1},
            {"flight_no": "FD3180", "departure": "13:00", "arrival": "03:30+1", "duration": "14h 30m", "airline": "AirAsia", "stops": 1},
            {"flight_no": "EK376", "departure": "22:15", "arrival": "14:45+1", "duration": "16h 30m", "airline": "Emirates", "stops": 1},
        ]
    },
    "maldives": {
        "price": "$1500",
        "flights": [
            {"flight_no": "MV101", "departure": "09:00", "arrival": "22:00", "duration": "13h 00m", "airline": "Maldivian", "stops": 1},
            {"flight_no": "EK650", "departure": "14:30", "arrival": "05:30+1", "duration": "15h 00m", "airline": "Emirates", "stops": 1},
            {"flight_no": "SL402", "departure": "22:30", "arrival": "14:30+1", "duration": "16h 00m", "airline": "SriLankan Airlines", "stops": 1},
        ]
    },
    "istanbul": {
        "price": "$640",
        "flights": [
            {"flight_no": "TK001", "departure": "06:30", "arrival": "12:00", "duration": "5h 30m", "airline": "Turkish Airlines", "stops": 0},
            {"flight_no": "PC614", "departure": "13:00", "arrival": "19:00", "duration": "6h 00m", "airline": "Pegasus", "stops": 0},
            {"flight_no": "EK1711", "departure": "21:45", "arrival": "04:15+1", "duration": "6h 30m", "airline": "Emirates", "stops": 1},
        ]
    },
    "athens": {
        "price": "$610",
        "flights": [
            {"flight_no": "A3601", "departure": "07:00", "arrival": "13:00", "duration": "6h 00m", "airline": "Aegean Air", "stops": 0},
            {"flight_no": "OA301", "departure": "12:30", "arrival": "19:30", "duration": "7h 00m", "airline": "Olympic Air", "stops": 1},
            {"flight_no": "TK1874", "departure": "20:15", "arrival": "04:00+1", "duration": "7h 45m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "vienna": {
        "price": "$690",
        "flights": [
            {"flight_no": "OS064", "departure": "08:15", "arrival": "14:00", "duration": "5h 45m", "airline": "Austrian Airlines", "stops": 0},
            {"flight_no": "LH1348", "departure": "14:30", "arrival": "21:00", "duration": "6h 30m", "airline": "Lufthansa", "stops": 1},
            {"flight_no": "TK1884", "departure": "22:00", "arrival": "05:00+1", "duration": "7h 00m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "prague": {
        "price": "$600",
        "flights": [
            {"flight_no": "OK501", "departure": "07:30", "arrival": "13:15", "duration": "5h 45m", "airline": "Czech Airlines", "stops": 0},
            {"flight_no": "LH3388", "departure": "13:00", "arrival": "19:30", "duration": "6h 30m", "airline": "Lufthansa", "stops": 1},
            {"flight_no": "FR8804", "departure": "21:00", "arrival": "04:00+1", "duration": "7h 00m", "airline": "Ryanair", "stops": 1},
        ]
    },
    "budapest": {
        "price": "$580",
        "flights": [
            {"flight_no": "MA601", "departure": "08:00", "arrival": "13:30", "duration": "5h 30m", "airline": "Malev", "stops": 0},
            {"flight_no": "W62801", "departure": "14:00", "arrival": "20:15", "duration": "6h 15m", "airline": "Wizz Air", "stops": 1},
            {"flight_no": "TK1882", "departure": "22:15", "arrival": "05:00+1", "duration": "6h 45m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "warsaw": {
        "price": "$560",
        "flights": [
            {"flight_no": "LO281", "departure": "07:45", "arrival": "13:00", "duration": "5h 15m", "airline": "LOT Polish", "stops": 0},
            {"flight_no": "LH1626", "departure": "13:30", "arrival": "19:30", "duration": "6h 00m", "airline": "Lufthansa", "stops": 1},
            {"flight_no": "TK1892", "departure": "21:30", "arrival": "04:00+1", "duration": "6h 30m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "copenhagen": {
        "price": "$910",
        "flights": [
            {"flight_no": "SK901", "departure": "09:00", "arrival": "15:00", "duration": "6h 00m", "airline": "SAS", "stops": 0},
            {"flight_no": "DY626", "departure": "14:15", "arrival": "21:00", "duration": "6h 45m", "airline": "Norwegian", "stops": 0},
            {"flight_no": "BA1488", "departure": "21:30", "arrival": "05:00+1", "duration": "7h 30m", "airline": "British Airways", "stops": 1},
        ]
    },
    "stockholm": {
        "price": "$940",
        "flights": [
            {"flight_no": "SK404", "departure": "08:30", "arrival": "15:00", "duration": "6h 30m", "airline": "SAS", "stops": 0},
            {"flight_no": "DY4001", "departure": "14:00", "arrival": "21:15", "duration": "7h 15m", "airline": "Norwegian", "stops": 0},
            {"flight_no": "LH984", "departure": "22:00", "arrival": "06:00+1", "duration": "8h 00m", "airline": "Lufthansa", "stops": 1},
        ]
    },
    "oslo": {
        "price": "$960",
        "flights": [
            {"flight_no": "DY7001", "departure": "09:15", "arrival": "16:00", "duration": "6h 45m", "airline": "Norwegian", "stops": 0},
            {"flight_no": "SK480", "departure": "15:00", "arrival": "22:30", "duration": "7h 30m", "airline": "SAS", "stops": 0},
            {"flight_no": "BA768", "departure": "21:45", "arrival": "06:00+1", "duration": "8h 15m", "airline": "British Airways", "stops": 1},
        ]
    },
    "helsinki": {
        "price": "$920",
        "flights": [
            {"flight_no": "AY831", "departure": "08:00", "arrival": "15:00", "duration": "7h 00m", "airline": "Finnair", "stops": 0},
            {"flight_no": "SK8762", "departure": "14:30", "arrival": "22:15", "duration": "7h 45m", "airline": "SAS", "stops": 1},
            {"flight_no": "LH820", "departure": "22:15", "arrival": "06:30+1", "duration": "8h 15m", "airline": "Lufthansa", "stops": 1},
        ]
    },
    "zurich": {
        "price": "$980",
        "flights": [
            {"flight_no": "LX016", "departure": "09:30", "arrival": "15:30", "duration": "6h 00m", "airline": "Swiss Int'l Air", "stops": 0},
            {"flight_no": "LH4608", "departure": "15:00", "arrival": "22:00", "duration": "7h 00m", "airline": "Lufthansa", "stops": 1},
            {"flight_no": "TK1922", "departure": "21:30", "arrival": "05:30+1", "duration": "8h 00m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "geneva": {
        "price": "$970",
        "flights": [
            {"flight_no": "LX026", "departure": "08:45", "arrival": "14:45", "duration": "6h 00m", "airline": "Swiss Int'l Air", "stops": 0},
            {"flight_no": "EK038", "departure": "14:00", "arrival": "21:15", "duration": "7h 15m", "airline": "Emirates", "stops": 1},
            {"flight_no": "TK1930", "departure": "22:00", "arrival": "05:45+1", "duration": "7h 45m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "madrid": {
        "price": "$630",
        "flights": [
            {"flight_no": "IB3166", "departure": "07:00", "arrival": "12:30", "duration": "5h 30m", "airline": "Iberia", "stops": 0},
            {"flight_no": "VY6202", "departure": "13:15", "arrival": "19:30", "duration": "6h 15m", "airline": "Vueling", "stops": 0},
            {"flight_no": "UX4122", "departure": "20:45", "arrival": "03:30+1", "duration": "6h 45m", "airline": "Air Europa", "stops": 1},
        ]
    },
    "lisbon": {
        "price": "$620",
        "flights": [
            {"flight_no": "TP351", "departure": "07:30", "arrival": "13:00", "duration": "5h 30m", "airline": "TAP Air Portugal", "stops": 0},
            {"flight_no": "FR1296", "departure": "13:00", "arrival": "19:15", "duration": "6h 15m", "airline": "Ryanair", "stops": 1},
            {"flight_no": "TK1890", "departure": "21:15", "arrival": "04:00+1", "duration": "6h 45m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "dublin": {
        "price": "$700",
        "flights": [
            {"flight_no": "EI104", "departure": "08:00", "arrival": "13:30", "duration": "5h 30m", "airline": "Aer Lingus", "stops": 0},
            {"flight_no": "FR680", "departure": "14:00", "arrival": "20:15", "duration": "6h 15m", "airline": "Ryanair", "stops": 1},
            {"flight_no": "BA834", "departure": "22:00", "arrival": "05:00+1", "duration": "7h 00m", "airline": "British Airways", "stops": 1},
        ]
    },
    "edinburgh": {
        "price": "$690",
        "flights": [
            {"flight_no": "EI292", "departure": "07:15", "arrival": "12:45", "duration": "5h 30m", "airline": "Aer Lingus", "stops": 0},
            {"flight_no": "BA1450", "departure": "13:30", "arrival": "19:45", "duration": "6h 15m", "airline": "British Airways", "stops": 0},
            {"flight_no": "FR9204", "departure": "21:00", "arrival": "04:00+1", "duration": "7h 00m", "airline": "Ryanair", "stops": 1},
        ]
    },
    "cape_town": {
        "price": "$820",
        "flights": [
            {"flight_no": "SA234", "departure": "08:00", "arrival": "21:30", "duration": "13h 30m", "airline": "South African Airways", "stops": 0},
            {"flight_no": "EK771", "departure": "14:30", "arrival": "07:00+1", "duration": "16h 30m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR1367", "departure": "22:00", "arrival": "16:30+1", "duration": "18h 30m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "johannesburg": {
        "price": "$780",
        "flights": [
            {"flight_no": "SA224", "departure": "09:00", "arrival": "21:30", "duration": "12h 30m", "airline": "South African Airways", "stops": 0},
            {"flight_no": "EK761", "departure": "15:30", "arrival": "07:30+1", "duration": "16h 00m", "airline": "Emirates", "stops": 1},
            {"flight_no": "TK44", "departure": "22:30", "arrival": "16:00+1", "duration": "17h 30m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "rio_de_janeiro": {
        "price": "$880",
        "flights": [
            {"flight_no": "LA8080", "departure": "10:00", "arrival": "01:30+1", "duration": "15h 30m", "airline": "LATAM", "stops": 0},
            {"flight_no": "TP30", "departure": "16:00", "arrival": "08:30+1", "duration": "16h 30m", "airline": "TAP Air Portugal", "stops": 1},
            {"flight_no": "AF460", "departure": "23:00", "arrival": "17:30+1", "duration": "18h 30m", "airline": "Air France", "stops": 1},
        ]
    },
    "sao_paulo": {
        "price": "$860",
        "flights": [
            {"flight_no": "LA8012", "departure": "10:30", "arrival": "01:00+1", "duration": "14h 30m", "airline": "LATAM", "stops": 0},
            {"flight_no": "TP40", "departure": "16:30", "arrival": "07:30+1", "duration": "15h 00m", "airline": "TAP Air Portugal", "stops": 1},
            {"flight_no": "AF464", "departure": "22:00", "arrival": "15:00+1", "duration": "17h 00m", "airline": "Air France", "stops": 1},
        ]
    },
    "mexico_city": {
        "price": "$750",
        "flights": [
            {"flight_no": "AM402", "departure": "09:00", "arrival": "19:00", "duration": "10h 00m", "airline": "Aeromexico", "stops": 0},
            {"flight_no": "AA940", "departure": "14:00", "arrival": "00:30+1", "duration": "10h 30m", "airline": "American Airlines", "stops": 0},
            {"flight_no": "UA2020", "departure": "21:00", "arrival": "09:00+1", "duration": "12h 00m", "airline": "United Airlines", "stops": 1},
        ]
    },
    "cairo": {
        "price": "$670",
        "flights": [
            {"flight_no": "MS777", "departure": "08:00", "arrival": "15:30", "duration": "7h 30m", "airline": "EgyptAir", "stops": 0},
            {"flight_no": "EK927", "departure": "14:15", "arrival": "22:45", "duration": "8h 30m", "airline": "Emirates", "stops": 1},
            {"flight_no": "TK694", "departure": "21:45", "arrival": "07:00+1", "duration": "9h 15m", "airline": "Turkish Airlines", "stops": 1},
        ]
    },
    "mumbai": {
        "price": "$500",
        "flights": [
            {"flight_no": "AI101", "departure": "08:15", "arrival": "18:30", "duration": "10h 15m", "airline": "Air India", "stops": 0},
            {"flight_no": "EK500", "departure": "14:30", "arrival": "01:45+1", "duration": "11h 15m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR554", "departure": "22:00", "arrival": "11:00+1", "duration": "13h 00m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "delhi": {
        "price": "$480",
        "flights": [
            {"flight_no": "AI111", "departure": "07:45", "arrival": "17:30", "duration": "9h 45m", "airline": "Air India", "stops": 0},
            {"flight_no": "EK512", "departure": "14:00", "arrival": "00:30+1", "duration": "10h 30m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR564", "departure": "22:30", "arrival": "10:30+1", "duration": "12h 00m", "airline": "Qatar Airways", "stops": 1},
        ]
    },
    "jaipur": {
        "price": "$450",
        "flights": [
            {"flight_no": "AI473", "departure": "09:00", "arrival": "20:30", "duration": "11h 30m", "airline": "Air India", "stops": 1},
            {"flight_no": "EK518", "departure": "15:00", "arrival": "03:30+1", "duration": "12h 30m", "airline": "Emirates", "stops": 1},
            {"flight_no": "QR574", "departure": "23:00", "arrival": "13:00+1", "duration": "14h 00m", "airline": "Qatar Airways", "stops": 2},
        ]
    },
}


In [49]:
def chat(message, history):
    print("User:", history)
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [43]:
def get_ticket_price(destination):
    """Return the ticket price for a given destination city."""
    dest = destination.lower().replace(" ", "_")
    return  flight_data.get(dest, {}).get("price", "Price not available")

def get_flights(destination: str) -> list:
    """Return all flight options for a destination."""
    dest = destination.lower().replace(" ", "_")
    return flight_data.get(dest, {}).get("flights", [])

def cheapest_flight(destination: str) -> dict | None:
    """Return the first (cheapest) flight listed for a destination."""
    flights = get_flights(destination)
    return flights[0] if flights else None

def nonstop_flights(destination: str) -> list:
    """Return only non-stop flights for a destination."""
    return [f for f in get_flights(destination) if f["stops"] == 0]

def flight_confirmation(destination: str, flight_no: str) -> str:
    """Return a confirmation message for a selected flight."""
    flights = get_flights(destination)
    for f in flights:
        if f["flight_no"] == flight_no:
            return f"Your flight {f['flight_no']} with {f['airline']} departing at {f['departure']} and arriving at {f['arrival']} has been booked. Total price: {get_ticket_price(destination)}."
    return "Flight not found."

In [44]:
get_ticket_price("delhi")

'$480'

In [46]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination"],
        "additionalProperties": False,
    },
    "returns": {
        "type": "string",
        "description": "The price of a return ticket to the destination city.",
    },
}
flight_options_function = {
    "name": "get_flights",
    "description": "Get list of flights available for given destination city with flight details like flight_no, arrival, departure, duration, air_line and stops",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination"],
        "additionalProperties": False,
    },
    "returns": {
        "type": "array",
        "description": "List of flights available for the given destination city.",
    },
}
cheapest_flight_function = {
    "name": "cheapest_flight",
    "description": "Get the cheapest flight available for a given destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            }
        },
        "required": ["destination"],
        "additionalProperties": False,
    },
    "returns": {
        "type": "object",
        "description": "The cheapest flight available for the given destination city."
    }
}
nonstop_flights_function = {
    "name": "nonstop_flights",
    "description": "Get list of nonstop flights available for a given destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            },
        },
        "required": ["destination"],
        "additionalProperties": False,
    },
    "returns": {
        "type": "array",
        "description": "List of nonstop flights available for the given destination city."
    }
}
flight_confirmation_function = {
    "name": "flight_confirmation",
    "description": "Books the flight and returns a successful booking message",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "flight_no": {
                "type": "string",
                "description": "The flight number of the selected flight",
            },
        },
        "required": ["destination", "flight_no"],
        "additionalProperties": False,
    },
    "returns": {
        "type": "string",
        "description": "A confirmation message or error message for the flight booking."
    }
}


In [47]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": flight_options_function},
    {"type": "function", "function": cheapest_flight_function},
    {"type": "function", "function": nonstop_flights_function},
    {"type": "function", "function": flight_confirmation_function},
]

tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination'],
    'additionalProperties': False},
   'returns': {'type': 'string',
    'description': 'The price of a return ticket to the destination city.'}}},
 {'type': 'function',
  'function': {'name': 'get_flights',
   'description': 'Get list of flights available for given destination city with flight details like flight_no, arrival, departure, duration, air_line and stops',
   'parameters': {'type': 'object',
    'properties': {'destination': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination'],
    'additionalProperties': False},
   'returns': {'type': 'array',
    'description': 'List of f

In [56]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print("**************++++++++++++++((((((((()))))))))")
    print("Response:", response.choices[0])
    print("||||||||||||||||||||||||||||||||||||||||||||||")
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        print("**************++++++++++++++((((((((()))))))))")
        print("Messages after tool calls:", messages)
        print("||||||||||||||||||||||||||||||||||||||||||||||")
        messages.extend(responses)
        print("**************++++++++++++++((((((((()))))))))")
        print("Messages after adding tool responses:", messages)
        print("||||||||||||||||||||||||||||||||||||||||||||||")

        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [57]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments or "{}")
        except json.JSONDecodeError:
            arguments = {}
        try:
            if name == "get_ticket_price":
                destination = arguments.get("destination", "")
                result = get_ticket_price(destination)

            elif name == "get_flights":
                destination = arguments.get("destination", "")
                result = get_flights(destination)

            elif name == "cheapest_flight":
                destination = arguments.get("destination", "")
                result = cheapest_flight(destination)

            elif name == "nonstop_flights":
                destination = arguments.get("destination", "")
                result = nonstop_flights(destination)

            elif name == "flight_confirmation":
                destination = arguments.get("destination", "")
                flight_no = arguments.get("flight_no", "")
                result = flight_confirmation(destination, flight_no)

            else:
                result = f"Unknown tool: {name}"

        except Exception as e:
            result = f"Tool execution error in {name}: {str(e)}"

        if isinstance(result, (dict, list)):
            content = json.dumps(result, ensure_ascii=False)
        elif result is None:
            content = "null"
        else:
            content = str(result)

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        })

    print("**************++++++++++++++((((((((()))))))))")
    print("Tool responses:", responses)
    print("||||||||||||||||||||||||||||||||||||||||||||||")
    return responses

In [59]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


**************++++++++++++++((((((((()))))))))
Response: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pZtw7UYCJcDMnuN8QXC3bhQA', function=Function(arguments='{"destination": "Delhi"}', name='get_ticket_price'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_IIQfwmgsvbQiAF6Dk7abLshZ', function=Function(arguments='{"destination": "Mumbai"}', name='get_ticket_price'), type='function')]))
||||||||||||||||||||||||||||||||||||||||||||||
**************++++++++++++++((((((((()))))))))
Tool responses: [{'role': 'tool', 'content': '$480', 'tool_call_id': 'call_pZtw7UYCJcDMnuN8QXC3bhQA'}, {'role': 'tool', 'content': '$500', 'tool_call_id': 'call_IIQfwmgsvbQiAF6Dk7abLshZ'}]
||||||||||||||||||||||||||||||||||||||||||||||
**************++++++++++++++((((((((()))))))))
Messages after tool 